# 02_Data_Understanding

Notebook ini mendokumentasikan fase Data Understanding untuk proyek prediksi harga komoditas pertanian Kabupaten Sumbawa. Fokus utama adalah inventarisasi data, profil data, dan quality check dari dataset yang tersedia.

## Aktivasi Environment RAPIDS

Sebelum menjalankan analisis ini, gunakan environment conda RAPIDS yang bernama `rapids-24.10`. Contoh perintah:

```bash
conda env create -f environment.yml
conda activate rapids-24.10
python -c "import cudf, cupy, torch; print('RAPIDS', cudf.__version__, 'CuPy', cupy.__version__, 'Torch', torch.__version__)"
```


## Inventaris Data Mentah

Data mentah tersedia di folder `data/raw/`. Selain data harga komoditas harian, terdapat juga dataset produktivitas triwulanan. Fase ini akan menjawab: file apa saja yang tersedia, berapa banyak komoditas, rentang tanggal, dan struktur kolom.

In [15]:
import pandas as pd
from pathlib import Path

raw_dir = Path('../data/raw')
files = sorted([p.name for p in raw_dir.iterdir() if p.is_file()])
print('Files tersedia di data/raw/')
for fname in files:
    print('-', fname)

price_path = raw_dir / 'DATASET HARGA KOMODITI.csv'
prod_path = raw_dir / 'produksi.csv'

price_df = pd.read_csv(price_path)
price_df.columns = [col.strip() for col in price_df.columns]
price_df['Tanggal'] = pd.to_datetime(price_df['Tanggal'], errors='coerce')

print('\nRincian DATASET HARGA KOMODITI.csv:')
print('Jumlah baris:', len(price_df))
print('Kolom:', list(price_df.columns))
print('Rentang tanggal:', price_df['Tanggal'].min(), 'sampai', price_df['Tanggal'].max())
print('Jumlah nilai tanggal yang tidak valid:', price_df['Tanggal'].isna().sum())
print('Unique komoditas:', sorted(price_df['Komoditi'].dropna().unique()))
print('Frekuensi per komoditas:')
print(price_df['Komoditi'].value_counts())

prod_df = pd.read_csv(prod_path)
print('\nRincian produksi.csv:')
print('Jumlah baris:', len(prod_df))
print('Kolom:', list(prod_df.columns))
print('Komoditas di data produksi:', sorted(prod_df['Komoditi'].dropna().unique()))
print(prod_df.groupby('Komoditi').agg({'tahun': 'nunique', 'triwulan': 'nunique'}))


Files tersedia di data/raw/
- DATASET HARGA KOMODITI.csv
- produksi.csv

Rincian DATASET HARGA KOMODITI.csv:
Jumlah baris: 4258
Kolom: ['Komoditi', 'Tanggal', 'Harga Petani', 'Harga Pengecer']
Rentang tanggal: 2022-01-01 00:00:00 sampai 2024-11-30 00:00:00
Jumlah nilai tanggal yang tidak valid: 1
Unique komoditas: ['Beras Medium', 'Beras Premium', 'Jagung Pipil Kering', 'Kacang Hijau']
Frekuensi per komoditas:
Komoditi
Beras Medium           1065
Beras Premium          1065
Kacang Hijau           1065
Jagung Pipil Kering    1063
Name: count, dtype: int64

Rincian produksi.csv:
Jumlah baris: 30
Kolom: ['Komoditi', 'tahun', 'triwulan', 'Luas Panen', 'Produktivitas', 'Produksi']
Komoditas di data produksi: ['Padi sawah', 'jagung', 'kacang hijau', 'padi gogo', 'padi total']
              tahun  triwulan
Komoditi                     
Padi sawah        2         3
jagung            2         3
kacang hijau      2         3
padi gogo         2         3
padi total        2         3


## Profil Data Harga Komoditas

Dataset harga menunjukkan beberapa karakteristik penting yang harus ditangani sebelum pemodelan. Pada data awal ditemukan: format harga berada dalam string dengan simbol mata uang, ada 1 nilai `Tanggal` tidak valid, dan terdapat missing values di kolom harga.

In [16]:
price_df['Komoditi'] = price_df['Komoditi'].astype(str).str.strip()
price_df['Harga Petani'] = pd.to_numeric(price_df['Harga Petani'].astype(str).str.replace('[^0-9.]', '', regex=True), errors='coerce')
price_df['Harga Pengecer'] = pd.to_numeric(price_df['Harga Pengecer'].astype(str).str.replace('[^0-9.]', '', regex=True), errors='coerce')

print('Jumlah missing per kolom harga:')
print(price_df[['Harga Petani', 'Harga Pengecer']].isna().sum())

print('Contoh record setelah pembersihan:')
print(price_df.head(5).to_string(index=False))


Jumlah missing per kolom harga:
Harga Petani      2158
Harga Pengecer    2157
dtype: int64
Contoh record setelah pembersihan:
           Komoditi    Tanggal  Harga Petani  Harga Pengecer
Jagung Pipil Kering 2022-01-03        4800.0          5000.0
Jagung Pipil Kering 2022-01-04        4800.0          5000.0
Jagung Pipil Kering 2022-01-05        4800.0          5000.0
Jagung Pipil Kering 2022-01-06        4800.0          5000.0
Jagung Pipil Kering 2022-01-07        4800.0          5000.0


## Profil Data Produktivitas

Dataset `produksi.csv` menyediakan informasi triwulanan untuk berbagai komoditas. Temuan penting pada data ini: ada 5 kategori komoditas, termasuk `jagung` dan `kacang hijau`, yang langsung relevan dengan model, serta beberapa entri `padi` yang dapat digunakan sebagai proxy untuk komoditas beras jika diperlukan.

In [17]:
print(prod_df.head(10).to_string(index=False))
print('Nilai missing per kolom produksi:')
print(prod_df.isna().sum())


  Komoditi  tahun  triwulan  Luas Panen  Produktivitas      Produksi
Padi sawah   2022         1  35685.6200      56.593625 201957.860000
Padi sawah   2022         2  12599.0000      54.279752  68387.060000
Padi sawah   2022         3   6888.7897      55.240632  38054.110000
Padi sawah   2023         1  26790.3606      55.977470 149965.660000
Padi sawah   2023         2  21687.8044      58.797478 127518.820000
Padi sawah   2023         3   8101.2824      54.176608  43890.000000
 padi gogo   2022         1   2871.0000      37.925865  10888.515770
 padi gogo   2022         2      0.0000       0.000000      0.000000
 padi gogo   2022         3      0.0000       0.000000      0.000000
 padi gogo   2023         1   1175.0000      38.176791   4485.772933
Nilai missing per kolom produksi:
Komoditi         0
tahun            0
triwulan         0
Luas Panen       0
Produktivitas    0
Produksi         0
dtype: int64


## Insight dari Data Understanding

- Data harga komoditas harian mencakup 4 seri yang siap dianalisis: Beras Premium, Beras Medium, Jagung Pipil Kering, dan Kacang Hijau.
- Rentang tanggal lengkap dari 2022-01-01 hingga 2024-11-30, dengan hanya satu nilai tanggal tidak valid.
- Harga masih berada dalam format string dan perlu diubah menjadi numerik.
- Terdapat missing values signifikan pada harga, sehingga fase selanjutnya harus mencakup strategi imputasi atau penghapusan observasi.
- Data produktivitas triwulanan tersedia dan dapat menjadi fitur eksogen, tetapi pemetaan langsung untuk beras perlu mempertimbangkan `padi` sebagai proxy.
- Environment conda yang digunakan untuk analisis ini adalah `rapids-24.10`.